# 1. Importing Faker 

In [ ]:
pip install faker # the liabrary which is build for the soul purpose of generating synthetic datasets

# 2. Code Generation Script

In [ ]:
import random
from datetime import datetime
import numpy as np
import pandas as pd
from faker import Faker

fake = Faker("en_IN")

Faker.seed(42)
np.random.seed(42)
random.seed(42)

TARGET_ROWS = 50000
BASE_ROWS = 48500

categories = [
    "Electronics",
    "Clothing & Apparel",
    "Home & Kitchen",
    "Beauty & Personal Care",
    "Books",
    "Sports & Fitness",
]
brands = [
    "Samsung",
    "Apple",
    "boAt",
    "Nike",
    "Adidas",
    "Puma",
    "Philips",
    "Sony",
    "Prestige",
    "LG",
    "Xiaomi",
    "Realme",
]
states = [
    "Maharashtra",
    "Delhi",
    "Karnataka",
    "Tamil Nadu",
    "Uttar Pradesh",
    "Gujarat",
    "West Bengal",
    "Telangana",
]

product_pool = []
for i in range(1000):
    brand = random.choice(brands)
    cat = random.choice(categories)
    product_pool.append(
        {
            "product_id": f"PROD-{10000 + i}",
            "product_name": f"{brand} {fake.word().capitalize()} {fake.word().capitalize()}",
            "category": cat,
            "brand": brand,
            "base_price": round(random.uniform(199, 49999), 2),
        }
    )

customer_pool = [f"CUST-{20000 + i}" for i in range(8000)]
data = []

for i in range(BASE_ROWS):
    prod = random.choice(product_pool)
    data.append(
        [
            f"ORD-{100000 + i}",
            fake.date_between(start_date="-2y", end_date="today").strftime(
                "%Y-%m-%d"
            ),
            random.choice(customer_pool),
            random.choice(states),
            prod["product_id"],
            prod["product_name"],
            prod["category"],
            prod["brand"],
            str(prod["base_price"]),
            random.choice([1, 1, 1, 2, 2, 3, 4, 5]),
            round(random.choice([0, 5, 10, 15, 20, 30, 50]), 2),
            random.choice(
                [
                    "Credit Card",
                    "Debit Card",
                    "UPI",
                    "Cash on Delivery",
                    "Net Banking",
                ]
            ),
            round(random.uniform(1.0, 5.0), 1),
            random.randint(0, 3500),
            random.choice(
                ["Delivered", "Shipped", "Cancelled", "Returned", "Pending"]
            ),
        ]
    )

columns = [
    "order_id",
    "order_date",
    "customer_id",
    "customer_state",
    "product_id",
    "product_name",
    "category",
    "brand",
    "price",
    "quantity",
    "discount_percent",
    "payment_method",
    "rating",
    "review_count",
    "delivery_status",
]
df = pd.DataFrame(data, columns=columns)

# Duplicate rows
duplicate_rows = df.sample(n=TARGET_ROWS - BASE_ROWS, random_state=42)
df = pd.concat([df, duplicate_rows], ignore_index=True)

# Unstandardized text casing & spacing
for col in [
    "category",
    "brand",
    "customer_state",
    "payment_method",
    "delivery_status",
]:
    df.loc[np.random.rand(len(df)) < 0.15, col] = (
        df[col].astype(str).str.lower()
    )
    df.loc[np.random.rand(len(df)) < 0.10, col] = (
        df[col].astype(str).str.upper()
    )
    df.loc[np.random.rand(len(df)) < 0.12, col] = "  " + df[col].astype(str) + " "

# Payment method variations
payment_replacements = {
    "UPI": ["upi", "U.P.I.", "UPI_PAY"],
    "Cash on Delivery": ["COD", "cod", "C.O.D."],
    "Credit Card": ["CC", "credit_card"],
}
for clean_val, dirty_list in payment_replacements.items():
    mask = (df["payment_method"].str.strip() == clean_val) & (
        np.random.rand(len(df)) < 0.35
    )
    df.loc[mask, "payment_method"] = np.random.choice(
        dirty_list, size=mask.sum()
    )


# Varied date formats
def randomize_date_format(date_str):
    if pd.isna(date_str) or random.random() > 0.25:
        return date_str
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return dt.strftime(random.choice(["%d/%m/%Y", "%m-%d-%Y", "%d-%b-%Y"]))


df["order_date"] = df["order_date"].apply(randomize_date_format)


# Dirty currency formats
def dirty_price(price_str):
    if random.random() < 0.15:
        return f"₹{price_str}"
    elif random.random() < 0.05:
        return f"INR {price_str}"
    return price_str


df["price"] = df["price"].apply(dirty_price)

# Quantity outliers
qty_mask = np.random.rand(len(df)) < 0.02
df.loc[qty_mask, "quantity"] = np.random.choice([-1, 0, 99], size=qty_mask.sum())

# Rating string noise (FIX: Explicitly cast to object dtype first)
df["rating"] = df["rating"].astype(object)
rating_mask = np.random.rand(len(df)) < 0.03
df.loc[rating_mask, "rating"] = "5.0 Stars"

# Null values
null_ratios = {
    "customer_state": 0.06,
    "brand": 0.04,
    "price": 0.05,
    "payment_method": 0.03,
    "rating": 0.08,
    "delivery_status": 0.02,
    "order_date": 0.03,
}
for col, ratio in null_ratios.items():
    df.loc[np.random.rand(len(df)) < ratio, col] = np.nan

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv("amazon_flipkart_raw_products.csv", index=False)

print(
    f"Successfully generated 'amazon_india_raw_products.csv' with {len(df):,} rows!"
)

Successfully generated 'amazon_flipkart_raw_products.csv' with 50,000 rows!


# 3. Understanding the Dataset

In [3]:
df = pd.read_csv("amazon_india_raw_products.csv")

print(df.head(30).to_string())

      order_id   order_date customer_id    customer_state  product_id                 product_name                category       brand        price  quantity  discount_percent    payment_method rating  review_count delivery_status
0   ORD-133553   2025-11-30  CUST-21239               NaN  PROD-10476              LG Odio Numquam      Clothing & Apparel          LG     40398.62         4                15       Net Banking    3.7          2744        returned
1   ORD-109427   06-03-2025  CUST-26699     Uttar Pradesh  PROD-10464                 Apple Cum At             Electronics       apple     21706.45         1                20               UPI    1.4           583        Returned
2   ORD-100199   2026-02-20  CUST-25088        Tamil Nadu  PROD-10940        Philips Ducimus Error      Clothing & Apparel     philips      41965.0         1                30               UPI    2.5          3261       Delivered
3   ORD-112447  30-Dec-2024  CUST-21637         Telangana  PROD-10414       

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   order_id          50000 non-null  str  
 1   order_date        48523 non-null  str  
 2   customer_id       50000 non-null  str  
 3   customer_state    46927 non-null  str  
 4   product_id        50000 non-null  str  
 5   product_name      50000 non-null  str  
 6   category          50000 non-null  str  
 7   brand             48006 non-null  str  
 8   price             47567 non-null  str  
 9   quantity          50000 non-null  int64
 10  discount_percent  50000 non-null  int64
 11  payment_method    48550 non-null  str  
 12  rating            45969 non-null  str  
 13  review_count      50000 non-null  int64
 14  delivery_status   48987 non-null  str  
dtypes: int64(3), str(12)
memory usage: 5.7 MB


In [5]:
df.describe()

,quantity,discount_percent,review_count
count,50000.000000,50000.00000,50000.000000
mean,2.959460,18.53420,1755.203080
std,7.817414,15.71755,1008.323101
min,-1.000000,0.00000,0.000000
25%,1.000000,5.00000,886.000000
50%,2.000000,15.00000,1756.000000
75%,4.000000,30.00000,2628.000000
max,99.000000,50.00000,3500.000000


# 4. Checking for Duplicates

In [7]:
# Returns a Series with duplicate count for each column
df.apply(lambda col: col.duplicated().sum())

order_id             1500
order_date          47081
customer_id         42024
customer_state      49951
product_id          49000
product_name        49002
category            49964
brand               49929
price               47130
quantity            49992
discount_percent    49993
payment_method      49964
rating              49957
review_count        46499
delivery_status     49969
dtype: int64

In [ ]:
# Removing Duplicates form column

# 1. order_id --> cuz every order has to be unique
# And rest of the column should contain Duplicates or else we loose the necessary data

# 5. Removing Duplicates

In [ ]:
df = df.drop_duplicates(subset=["order_id"], keep="first")


In [ ]:
# Before Removing duplicates --> 50000

df["order_id"].count() # After removing duplicates -->  48500

np.int64(48500)

# 6. Standardization 

In [14]:
print(df.head(10).to_string())

     order_id   order_date customer_id    customer_state  product_id              product_name            category    brand        price  quantity  discount_percent payment_method rating  review_count delivery_status
0  ORD-133553   2025-11-30  CUST-21239               NaN  PROD-10476           LG Odio Numquam  Clothing & Apparel       LG     40398.62         4                15    Net Banking    3.7          2744        returned
1  ORD-109427   06-03-2025  CUST-26699     Uttar Pradesh  PROD-10464              Apple Cum At         Electronics    apple     21706.45         1                20            UPI    1.4           583        Returned
2  ORD-100199   2026-02-20  CUST-25088        Tamil Nadu  PROD-10940     Philips Ducimus Error  Clothing & Apparel  philips      41965.0         1                30            UPI    2.5          3261       Delivered
3  ORD-112447  30-Dec-2024  CUST-21637         Telangana  PROD-10414    Apple Aliquam Possimus         Electronics    Apple     1189

In [17]:
# Standardize the Column 

# "customer_state", "delivery_status"

df["customer_state"] = df["customer_state"].str.strip().str.title()
df["delivery_status"] = df["delivery_status"].str.strip().str.title()



print(df.head(10).to_string())

     order_id   order_date customer_id customer_state  product_id              product_name            category    brand        price  quantity  discount_percent payment_method rating  review_count delivery_status
0  ORD-133553   2025-11-30  CUST-21239            NaN  PROD-10476           LG Odio Numquam  Clothing & Apparel       LG     40398.62         4                15    Net Banking    3.7          2744        Returned
1  ORD-109427   06-03-2025  CUST-26699  Uttar Pradesh  PROD-10464              Apple Cum At         Electronics    apple     21706.45         1                20            UPI    1.4           583        Returned
2  ORD-100199   2026-02-20  CUST-25088     Tamil Nadu  PROD-10940     Philips Ducimus Error  Clothing & Apparel  philips      41965.0         1                30            UPI    2.5          3261       Delivered
3  ORD-112447  30-Dec-2024  CUST-21637      Telangana  PROD-10414    Apple Aliquam Possimus         Electronics    Apple     11892.04         4 

# 7. Handling NaN (Missing Values)

In [20]:
missing = df.isna().sum()

missing

order_id               0
order_date          1430
customer_id            0
customer_state      2989
product_id             0
product_name           0
category               0
brand               1928
price               2360
quantity               0
discount_percent       0
payment_method      1406
rating              3911
review_count           0
delivery_status      979
dtype: int64

In [ ]:
# Handling the Critical Columns first

df = df.dropna(subset=["order_date", "price"])

In [22]:
missing = df.isna().sum()

missing

order_id               0
order_date             0
customer_id            0
customer_state      2755
product_id             0
product_name           0
category               0
brand               1780
price                  0
quantity               0
discount_percent       0
payment_method      1307
rating              3603
review_count           0
delivery_status      909
dtype: int64

In [23]:
# Filling the nan in textual column wilth "unknown"

text_column = ["customer_state", "brand", "payment_method", "delivery_status"]
df[text_column] = df[text_column].fillna("unknown")

In [24]:
missing = df.isna().sum()

missing

order_id               0
order_date             0
customer_id            0
customer_state         0
product_id             0
product_name           0
category               0
brand                  0
price                  0
quantity               0
discount_percent       0
payment_method         0
rating              3603
review_count           0
delivery_status        0
dtype: int64

In [25]:
# 1. Strip any leftover text (e.g., '4.5 Stars' -> '4.5') and convert to float
df["rating"] = pd.to_numeric(
    df["rating"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce",
)

# 2. Calculate median (ignores existing NaNs automatically)
rating_median = df["rating"].median()

# 3. Fill missing values with the calculated median
df["rating"] = df["rating"].fillna(rating_median)

In [26]:
missing = df.isna().sum()

missing

order_id            0
order_date          0
customer_id         0
customer_state      0
product_id          0
product_name        0
category            0
brand               0
price               0
quantity            0
discount_percent    0
payment_method      0
rating              0
review_count        0
delivery_status     0
dtype: int64

In [28]:
# 1. Strip all non-numeric characters except decimals
df["price"] = df["price"].astype(str).str.replace(r"[^\d.]", "", regex=True)

# 2. Convert to float cleanly
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Standardize all mixed date formats into standard datetime objects (YYYY-MM-DD)
df["order_date"] = pd.to_datetime(
    df["order_date"], format="mixed", dayfirst=True, errors="coerce"
)

In [29]:
# Check that data types changed from 'object' to 'datetime64[ns]' and 'float64'
print(df[["order_date", "price"]].dtypes)

# Preview the clean rows
df[["order_date", "price"]].head(10)

order_date    datetime64[us]
price                float64
dtype: object


,order_date,price
0,2025-11-30,40398.62
1,2025-03-06,21706.45
2,2026-02-20,41965.00
3,2024-12-30,11892.04
4,2025-02-20,36042.76
5,2026-08-23,12123.73
6,2025-08-04,2454.43
7,2025-08-03,3347.21
8,2026-02-13,20908.11
9,2026-01-03,9086.99


# 9. Changing the Data types

In [31]:
# quantity --> int64 to 

df = df.astype(
    {
        "quantity": "int64",
        "price": "float64",
        "customer_state": "string",
    }
)

In [32]:
df.info()

<class 'pandas.DataFrame'>
Index: 44775 entries, 0 to 49999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          44775 non-null  str           
 1   order_date        44775 non-null  datetime64[us]
 2   customer_id       44775 non-null  str           
 3   customer_state    44775 non-null  string        
 4   product_id        44775 non-null  str           
 5   product_name      44775 non-null  str           
 6   category          44775 non-null  str           
 7   brand             44775 non-null  str           
 8   price             44775 non-null  float64       
 9   quantity          44775 non-null  int64         
 10  discount_percent  44775 non-null  int64         
 11  payment_method    44775 non-null  str           
 12  rating            44775 non-null  float64       
 13  review_count      44775 non-null  int64         
 14  delivery_status   44775 non-null  str 

# 10. Feature Engineering

In [33]:
print(df.head().to_string())

     order_id order_date customer_id customer_state  product_id            product_name            category    brand     price  quantity  discount_percent payment_method  rating  review_count delivery_status
0  ORD-133553 2025-11-30  CUST-21239        unknown  PROD-10476         LG Odio Numquam  Clothing & Apparel       LG  40398.62         4                15    Net Banking     3.7          2744        Returned
1  ORD-109427 2025-03-06  CUST-26699  Uttar Pradesh  PROD-10464            Apple Cum At         Electronics    apple  21706.45         1                20            UPI     1.4           583        Returned
2  ORD-100199 2026-02-20  CUST-25088     Tamil Nadu  PROD-10940   Philips Ducimus Error  Clothing & Apparel  philips  41965.00         1                30            UPI     2.5          3261       Delivered
3  ORD-112447 2024-12-30  CUST-21637      Telangana  PROD-10414  Apple Aliquam Possimus         Electronics    Apple  11892.04         4                 5             C

In [36]:
conditions = [

    (df["price"] < 1000),
    (df["price"] >= 1000) & (df["price"] < 5000),
    (df["price"] >= 5000) & (df["price"] < 20000),
    (df["price"] >= 20000),
]

choices = ["Budget", "Economy", "Mid-Range", "Premium"]

df["price_category"] = np.select(conditions, choices, default="unknown")

# 11. Export the DataSet

In [38]:
df.to_csv("amazon_sales_cleaned.csv", index=False)